# 다중 필터 시계열 측광 (Light Curve) 분석 — Colab + 로컬 런타임

## 이 프로그램은 무엇을 하나요?

밤하늘을 여러 번 찍은 사진(FITS 파일)을 컴퓨터가 자동으로 분석해서, **어떤 별이 시간이 지나면서 얼마나 밝아지고 어두워지는지**를 그래프로 그려줍니다. 별 하나의 밝기를 사람이 사진 수십~수백 장마다 눈으로 재는 건 사실상 불가능하니, 이 과정을 코드로 자동화한 것입니다.

**전체 흐름을 한 문장으로**: 사진들을 열어서 → 별의 위치를 찾고 → 그 별의 밝기를 재고 → 시간 순서대로 늘어놓아 그래프로 그린다.

## 먼저 알아두면 좋은 용어들

이 노트북 안에서 계속 등장하는 단어들입니다. 처음 보면 낯설 수 있어서 미리 정리해둡니다.

| 용어 | 뜻 |
|---|---|
| **FITS 파일** | 천문학에서 쓰는 사진 파일 형식. 우리가 흔히 쓰는 jpg/png와 달리, 사진 안에 "언제, 어떤 필터로, 얼마나 노출을 줬는지" 같은 관측 정보(헤더)가 같이 저장되어 있습니다. |
| **필터 (g, r, i / B, V, R)** | 특정 색깔(파장)의 빛만 통과시키는 유리판. 카메라 앞에 다른 색 필터를 끼우고 찍으면, 그 색으로만 본 사진이 나옵니다. 같은 별도 필터마다 밝기가 조금씩 다르게 찍힙니다. |
| **등급(magnitude)** | 별의 밝기를 나타내는 숫자. **주의**: 숫자가 작을수록 더 밝은 별입니다 (거꾸로 되어 있어서 헷갈리기 쉬운데, 고대부터 내려온 관습입니다). |
| **구경측광(aperture photometry)** | 사진 위에 별을 중심으로 동그란 원을 그려서, **그 원 안에 들어온 빛을 전부 더하는 방법**. "이 원 안에 담긴 빛의 총량 = 이 별의 밝기"라고 보는 겁니다. |
| **배경환(annulus)** | 측광 원 바로 바깥의 도넛 모양 영역. 별빛이 거의 없는 순수한 밤하늘 배경이 얼마나 밝은지 재는 데 씁니다 — 이걸 알아야 별빛에서 배경빛을 빼서 "진짜 별빛"만 남길 수 있습니다. |
| **비교성(comparison star)** | 대상 별 근처에 있는, 밝기가 거의 변하지 않는다고 보는 다른 별. 대상 별과 비교성을 같이 재서 "대상이 비교성보다 얼마나 더/덜 밝아졌나"를 계산합니다. |
| **차등측광(differential photometry)** | 대상 별의 등급에서 비교성의 등급을 빼는 방법. 구름이 지나가거나 대기 상태가 바뀌면 두 별 다 똑같이 영향을 받기 때문에, 빼주면 그런 공통 오차가 상쇄되고 **대상 별 자체의 진짜 밝기 변화만** 남습니다. |
| **광도곡선(light curve)** | 시간을 가로축, 밝기(등급)를 세로축으로 그린 그래프. 이게 이 노트북의 최종 결과물입니다. |
| **gain(이득, e-/ADU)** | 카메라 센서가 빛 알갱이(광자)를 전기 신호로 바꾸는 효율을 나타내는 값. 측광 오차를 정확히 계산하려고 필요합니다. |
| **AIRMASS(대기질량)** | 별빛이 지구 대기를 얼마나 두껍게 통과해서 왔는지 나타내는 수치. 별이 지평선에 가까울수록(고도가 낮을수록) 이 값이 커지고, 대기를 오래 통과하면서 빛이 더 흐려집니다. |

## 이 프로그램이 하는 일 (6단계)

1. 각 사진의 **관측 시각**을 사진 파일(FITS 헤더)에서 읽어옵니다.
2. 사진마다 시야가 밀리거나 돌아간 정도를 계산해서 측광 원이 별을 따라가게 합니다 (**프레임 정렬**).
3. 대상 별과 비교성에 각각 **구경측광**을 해서 밝기(등급)를 잽니다.
4. **차등 등급 = 대상 등급 − 비교성 등급**을 계산합니다.
5. 필터별로 **광도곡선** 그래프를 그립니다.
6. (외계행성 식현상이면) 예보된 식 시각에 맞춰 **식 곡선 피팅**으로 식 깊이를 숫자로 구합니다.

**로컬 컴퓨터에 있는 FITS 파일들을 구글 드라이브에 업로드하지 않고 그대로 사용**합니다. Colab의 "로컬 런타임에 연결" 기능을 써서, 화면(UI)은 Colab 그대로 쓰면서 실제 코드 실행은 본인 컴퓨터에서 하도록 합니다. 용량 제한(구글 드라이브 15GB)이나 업로드 시간 걱정 없이, 원래 하던 대로 셀 실행 → 그래프 인라인 표시가 됩니다.

필터 이름이나 개수를 하드코딩하지 않아서 `g, r, i`든 `B, V, R`든, 필터를 몇 개 쓰든 `FILE_PATTERN_BY_FILTER` 설정만 바꾸면 그대로 동작합니다. 색지수 그래프(마지막 섹션)도 넣은 순서대로 인접한 필터끼리 자동으로 짝지어 계산합니다 (예: g,r,i 순서면 g-r, r-i).

## 사용 전 준비물
- **같은 대상을 같은 필터로 여러 번(시간 간격을 두고) 반복 촬영한 FITS 파일들.** 필터당 1장씩만 있으면 그 순간의 색만 알 수 있을 뿐 광도곡선은 나오지 않습니다.
- **이 노트북은 전처리(Bias/Dark/Flat 보정)를 하지 않습니다.** 미리 보정을 끝낸 라이트 프레임을 준비하세요.
  - 보정 후에도 헤더의 `DATE-OBS` 값이 그대로 남아있는지 확인하세요.
- **필터별로 폴더를 나눌 필요 없이, 한 폴더 안에 파일명 끝(`-g.fit`, `-r.fit`, `-i.fit` 등)으로 필터를 구분**하는 구조를 기본으로 가정합니다 (실제 확인된 구조: `pp_YZ Bootis_20260328-0001-g.fit`). 폴더를 나누는 구조를 쓰신다면 아래 [설정] 셀만 그에 맞게 바꾸면 됩니다.
- 각 FITS 헤더에 관측시각 키워드(`DATE-OBS` 등)가 있어야 시간축을 만들 수 있습니다.
- 대상이 **성단처럼 별이 촘촘한 시야**라면, 대상/비교성으로 쓸 별은 주변에 다른 별이 겹치지 않는 고립된 별로 골라야 측광이 오염되지 않습니다.

아래 셀들을 위에서부터 순서대로 실행하세요. **[설정]** 표시된 셀은 반드시 본인 데이터에 맞게 값을 수정해야 합니다.

In [ ]:
# 1. 필요한 패키지 설치 (로컬 런타임에 이미 설치돼 있다면 그냥 확인만 하고 넘어갑니다)
!pip install -q astropy photutils

## 2. 로컬 런타임 연결 (구글 드라이브 마운트 대신)

이 노트북은 구글 드라이브를 쓰지 않습니다. 대신 **본인 컴퓨터의 Jupyter 서버에 Colab을 연결**해서, 로컬 파일을 그대로 읽습니다. 아래는 컴퓨터 쪽에서 딱 한 번만 하면 되는 설정입니다 (이미 해두셨다면 3번 셀로 바로 넘어가세요).

**터미널(cmd/PowerShell)에서**:
```bash
pip install jupyter jupyter_http_over_ws
jupyter server extension enable --py jupyter_http_over_ws
jupyter notebook --NotebookApp.allow_origin="https://colab.research.google.com" --port=8888 --NotebookApp.port_retries=0
```
실행하면 `http://localhost:8888/?token=xxxxxxxx` 같은 URL이 출력됩니다.

**Colab 화면에서**: 오른쪽 위 "연결(Connect)" 버튼 옆 작은 화살표(▼) 클릭 → **"로컬 런타임에 연결"** 선택 → 위에서 나온 URL을 붙여넣기.

연결되면 이 아래 셀들에서 `C:\...` 같은 로컬 경로를 그대로 쓸 수 있습니다. astropy, photutils, numpy, pandas, matplotlib이 로컬 Python에 이미 설치되어 있다면 별도 준비 없이 바로 3번 셀부터 실행하면 됩니다.

In [ ]:
# 3. 라이브러리 임포트
import glob
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.time import Time
from astropy.stats import sigma_clipped_stats
from astropy.visualization import ZScaleInterval

from photutils.detection import DAOStarFinder
from photutils.aperture import CircularAperture, CircularAnnulus, aperture_photometry
from photutils.centroids import centroid_sources, centroid_com

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

## (그림으로 이해하기) 구경측광은 실제로 어떻게 생겼나

아래는 **가상의 별 하나**를 그려서, "측광 원(구경)"과 "배경환(annulus)"이 사진 위에서 실제로 어떤 모양으로 놓이는지 보여주는 그림입니다. 실제 데이터가 아니라 설명용으로 만든 그림이에요.

- **가운데 밝은 얼룩** = 별빛 (가장자리로 갈수록 흐려짐)
- **빨간 원** = 측광 원(구경) — 이 안의 빛을 다 더해서 별의 밝기를 잽니다
- **노란 점선 도넛** = 배경환 — 별빛이 없는 순수 배경을 재는 영역

`net_flux = (빨간 원 안의 빛 총합) − (노란 도넛에서 잰 배경 밝기) × (빨간 원의 넓이)` 이런 식으로 계산합니다 — 즉 "별+배경"에서 "배경만"을 빼서 "순수 별빛"을 남기는 것입니다.

In [ ]:
# 설명용 가상의 별 이미지를 만들고, 구경/배경환을 그려서 보여줍니다 (실제 데이터 아님)
_demo_size = 200
_yy, _xx = np.mgrid[0:_demo_size, 0:_demo_size]
_cx, _cy = _demo_size / 2, _demo_size / 2
_star_sigma = 8.0
_background_level = 100.0
rng = np.random.default_rng(0)
_demo_star = 4000 * np.exp(-(((_xx - _cx) ** 2 + (_yy - _cy) ** 2) / (2 * _star_sigma ** 2)))
_demo_image = _background_level + _demo_star + rng.normal(0, 5, size=(_demo_size, _demo_size))

_demo_aperture_r = 14
_demo_annulus_r_in = 20
_demo_annulus_r_out = 28

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(_demo_image, cmap='inferno', origin='lower')
ax.add_patch(plt.Circle((_cx, _cy), _demo_aperture_r, fill=False, color='red', lw=2, label='측광 원 (구경)'))
ax.add_patch(plt.Circle((_cx, _cy), _demo_annulus_r_in, fill=False, color='yellow', lw=1.5, ls='--'))
ax.add_patch(plt.Circle((_cx, _cy), _demo_annulus_r_out, fill=False, color='yellow', lw=1.5, ls='--', label='배경환 (annulus)'))
ax.set_title('가상의 별로 본 측광 원 & 배경환 (설명용, 실제 데이터 아님)')
ax.legend(loc='upper right', facecolor='white', framealpha=0.9)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## [설정] 데이터 경로와 측광 파라미터

- `BASE_DIR`: FITS 파일들이 들어있는 **로컬 폴더 경로** (필터별로 나뉘어 있지 않아도 됨).
- `FILE_PATTERN_BY_FILTER`: 필터별로 파일을 구분하는 **파일명 패턴**. 딕셔너리에 넣은 **순서가 그래프 색상 순서 및 색지수(예: g-r, r-i) 계산 순서**로 그대로 쓰입니다.
  - **필터 하나만 분석**하려면 딕셔너리에 그 필터 한 줄만 남기면 됩니다 (예: `{'g': '*-g.fit'}`). 파일이 없는 필터는 경고 후 자동으로 건너뜁니다.
- `ALIGN_ENABLED`, `ALIGN_NUM_STARS`: **프레임 정렬**(시야 이동/회전 자동 추적) 여부와 정렬에 쓸 별 개수. 켜두면(기본값) 밤새 시야가 밀리거나 돌아가도 측광 원이 별을 따라갑니다 (5.5절 참고).
- `COLOR_INDEX_ENABLED`: **색지수 그래프(10절)를 그릴지 여부** (옵션). 필터가 1개뿐이면 켜져 있어도 자동으로 건너뜁니다.
- `APERTURE_RADIUS`, `ANNULUS_R_IN`, `ANNULUS_R_OUT`: 측광 구경/배경환(annulus) 반지름 (단위: 픽셀). 아래 "(옵션) 구경 반지름 자동 최적화" 셀을 실행하면 실제 데이터 기준으로 자동 갱신됩니다.
- `GAIN_DEFAULT`: 카메라 이득(e-/ADU). 헤더의 `EGAIN` 값이 flat 프레임 실측값과 크게 달라 신뢰할 수 없어서, flat 2장으로 직접 측정한 값을 고정값으로 씁니다. 아래 "카메라 gain 측정" 셀로 언제든 재측정할 수 있습니다.
- `MAX_CENTROID_SHIFT_PX`, `MAX_DIFF_MAG_ERR_MULTIPLIER`, `MIN_DIFF_MAG_ERR_FLOOR`: **아웃라이어 프레임 자동 제거** 기준값.
- `TARGET_TYPE`: **대상이 어떤 현상인지 직접 지정** (`1` = 변광성, `2` = 외계행성 식현상/트랜싯). 이 값에 따라 `DETRENDING_ENABLED`/`BINNING_ENABLED` 기본값이 자동으로 정해집니다.
  - 변광성(1): 대상 자체가 관측 시간 내내 크게(때로는 몇 등급씩) 변할 수 있어서, 추세 제거/비닝이 진짜 신호를 깎아버릴 위험이 큽니다 → 둘 다 기본 꺼짐.
  - 외계행성 식현상(2): 신호가 아주 미세(보통 0.1~2%, 등급으로 0.001~0.02mag)해서 대기 소광 잔여 추세에 묻히기 쉽고, 점당 정밀도 확보가 더 중요합니다 → 둘 다 기본 켜짐.
  - 자동으로 정해진 값을 그대로 써도 되고, 필요하면 설정 셀 맨 아래에서 다시 덮어써도 됩니다.
- `DETRENDING_ENABLED`, `DETREND_DEGREE`: **추세 제거** 여부/차수 (기본값은 `TARGET_TYPE`에서 자동 결정).
- `BINNING_ENABLED`, `BIN_WIDTH_MINUTES`: **시간 구간 비닝** 여부/구간폭 (기본값은 `TARGET_TYPE`에서 자동 결정).

In [ ]:
# ==== 여기를 본인 데이터에 맞게 수정하세요 ====

BASE_DIR = r'C:\26astro\YZBootis\result\pp'  # FITS 파일들이 들어있는 로컬 폴더

# 필터를 몇 개 쓰든 원하는 이름/개수로 자유롭게 채우면 됩니다.
# 딕셔너리에 넣은 순서 = 그래프 색상 순서 = 색지수(예: g-r, r-i) 계산 순서.
FILE_PATTERN_BY_FILTER = {
    'g': '*-g.fit',
    'r': '*-r.fit',
    'i': '*-i.fit',
}
# 필터 하나만 분석하려면 그 필터 한 줄만 남기면 됩니다. 예: FILE_PATTERN_BY_FILTER = {'g': '*-g.fit'}
# (여기 적었는데 파일이 없는 필터는 측광 단계에서 경고만 출력하고 자동으로 건너뜁니다.)

# 측광 구경 설정 (픽셀 단위)
# "(옵션) 구경 반지름 자동 최적화" 셀을 실행하면 이 값들이 자동으로 갱신됩니다.
APERTURE_RADIUS = 12
ANNULUS_R_IN = 16
ANNULUS_R_OUT = 22

# 별 중심을 다시 찾을 때 탐색 반경 (이미지 사이 별 위치가 살짝 흔들리는 것을 보정)
CENTROID_SEARCH_BOX = 9

# 프레임 정렬: 밤새 시야가 서서히 밀리거나 돌아가도 측광 원이 별을 따라가게 합니다 (아래 5.5절 참고).
ALIGN_ENABLED = True
ALIGN_NUM_STARS = 14   # 정렬에 쓸 밝고 고립된 별 개수 (상위 후보 중에서 자동으로 고름)

# 여러 확장자/헤더 포맷 대응: 관측시각 헤더 키워드 후보 (앞에서부터 우선 탐색)
DATE_OBS_KEYS = ['DATE-OBS', 'DATE_OBS', 'DATEOBS']
TIME_OBS_KEYS = ['TIME-OBS', 'TIME_OBS', 'UT']  # DATE-OBS에 시각이 없을 때 보조로 사용

# 카메라 이득(e-/ADU) — flat-frame 실측값. 헤더 EGAIN은 신뢰 불가로 확인되어 사용하지 않음.
GAIN_DEFAULT = 0.71

# 아웃라이어 프레임 자동 제거 기준
MAX_CENTROID_SHIFT_PX = 6.0          # 대상/비교성 위치가 지정 좌표에서 이 값(px) 이상 벗어나면 추적 실패로 간주
MAX_DIFF_MAG_ERR_MULTIPLIER = 5.0    # 그 필터의 오차 중앙값의 이 배수를 넘으면 저품질로 간주
MIN_DIFF_MAG_ERR_FLOOR = 0.02        # 위 배수 기준의 절대 하한(mag)

# ---- 대상 유형 설정: 1 = 변광성, 2 = 외계행성 식현상(트랜싯) ----
# 이 값 하나로 아래 추세 제거/비닝 기본값이 자동으로 정해집니다.
TARGET_TYPE = 1

if TARGET_TYPE == 1:
    # 변광성: 진짜 신호(큰 폭의 변화)를 보존해야 하므로 추세 제거/비닝 기본 꺼짐
    DETRENDING_ENABLED = False
    BINNING_ENABLED = False
    print('대상 유형: 변광성 -> 추세 제거/비닝 기본 꺼짐 (큰 폭의 실제 변화를 보존)')
elif TARGET_TYPE == 2:
    # 외계행성 식현상: 신호가 미세(0.1~2%)해서 정밀도 확보가 우선 -> 추세 제거/비닝 기본 켜짐
    DETRENDING_ENABLED = True
    BINNING_ENABLED = True
    print('대상 유형: 외계행성 식현상 -> 추세 제거/비닝 기본 켜짐 (미세 신호 검출 정밀도 우선)')
else:
    raise ValueError('TARGET_TYPE은 1(변광성) 또는 2(외계행성 식현상) 중 하나여야 합니다.')

DETREND_DEGREE = 1   # 1=선형, 2=이차 다항식으로 추세를 피팅
BIN_WIDTH_MINUTES = 10.0   # 이 시간(분) 이내의 프레임들을 하나로 묶음

# ---- 필요하면 위 자동 설정값을 여기서 직접 덮어쓸 수 있습니다 ----
# DETRENDING_ENABLED = True
# BINNING_ENABLED = False

# 색지수(필터 간 색 차이) 그래프를 그릴지 여부. 필터가 1개뿐이면 True여도 자동으로 건너뜁니다.
COLOR_INDEX_ENABLED = True

# 그래프/색지수 계산에 쓰이는 필터 순서
FILTER_ORDER = list(FILE_PATTERN_BY_FILTER.keys())

print('설정 완료. 필터 순서:', FILTER_ORDER)

## (옵션) 카메라 gain 직접 측정

헤더의 `EGAIN`을 못 믿을 때(또는 카메라 게인 설정을 바꿨을 때) **같은 노출시간의 flat 프레임 2장**으로 직접 gain(e-/ADU)을 측정하는 셀입니다. mean-variance(광자전달곡선) 방법을 씁니다: 두 flat의 차이 이미지에는 광자잡음만 남고(고정 패턴은 상쇄됨), `gain = (평균1+평균2) / 차이이미지_분산`으로 계산됩니다.

- gain은 필터나 대상과 무관한 **카메라 하드웨어 특성**이라 한 번만 측정하면 계속 씁니다.
- 이 셀을 실행하면 위 설정 셀의 `GAIN_DEFAULT`가 자동으로 갱신됩니다.

In [ ]:
# 같은 노출시간으로 찍은 flat 프레임 2장의 로컬 경로를 지정하세요.
FLAT_FILE_1 = r'C:\26astro\For Preprocess\YZBootis\A\flat\flat_g_0001.fit'
FLAT_FILE_2 = r'C:\26astro\For Preprocess\YZBootis\A\flat\flat_g_0002.fit'
GAIN_PATCH_HALF = 200  # 중앙 패치 크기(픽셀 반폭). 비네팅/먼지 그림자를 피하려고 중앙의 작은 영역만 사용


def measure_gain(file1, file2, patch_half=GAIN_PATCH_HALF):
    d1 = fits.getdata(file1).astype(float)
    d2 = fits.getdata(file2).astype(float)
    if d1.shape != d2.shape:
        raise ValueError('두 flat 프레임의 크기가 다릅니다.')

    h, w = d1.shape
    cy, cx = h // 2, w // 2
    p1 = d1[cy - patch_half:cy + patch_half, cx - patch_half:cx + patch_half]
    p2 = d2[cy - patch_half:cy + patch_half, cx - patch_half:cx + patch_half]

    m1, _, _ = sigma_clipped_stats(p1, sigma=4.0)
    m2, _, _ = sigma_clipped_stats(p2, sigma=4.0)
    _, _, diff_std = sigma_clipped_stats(p1 - p2, sigma=4.0)

    gain = (m1 + m2) / diff_std**2
    return gain, m1, m2


try:
    measured_gain, flat_mean1, flat_mean2 = measure_gain(FLAT_FILE_1, FLAT_FILE_2)
    print(f'프레임1 평균 = {flat_mean1:.1f} ADU, 프레임2 평균 = {flat_mean2:.1f} ADU')
    print(f'측정된 gain = {measured_gain:.4f} e-/ADU')
    GAIN_DEFAULT = measured_gain  # 위 설정 셀의 값을 실측값으로 덮어씀
    print(f'GAIN_DEFAULT를 {GAIN_DEFAULT:.4f}(으)로 갱신했습니다.')
except (FileNotFoundError, OSError):
    print(f'flat 프레임을 찾지 못해 이 단계는 건너뜁니다 (GAIN_DEFAULT는 위 설정값 {GAIN_DEFAULT} 그대로 사용).')
    print('나중에 gain을 재측정하려면 FLAT_FILE_1/FLAT_FILE_2 경로를 실제 위치로 맞추고 다시 실행하세요.')

## 4. 공통 함수 정의

In [ ]:
def list_fits_files(base_dir, pattern):
    files = sorted(glob.glob(os.path.join(base_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"'{base_dir}' 에서 '{pattern}' 패턴의 파일을 찾지 못했습니다.")
    return files


def load_fits_data(path):
    """첫 번째로 실제 이미지 데이터가 들어있는 HDU를 찾아 (data, header) 반환."""
    with fits.open(path) as hdul:
        for hdu in hdul:
            if hdu.data is not None and hdu.data.ndim == 2:
                return hdu.data.astype(float), hdu.header
    raise ValueError(f"'{path}' 에서 2D 이미지 데이터를 찾지 못했습니다.")


def get_obs_time(header):
    """헤더에서 관측 시각(astropy Time)을 추출."""
    date_val = None
    for key in DATE_OBS_KEYS:
        if key in header:
            date_val = header[key]
            break
    if date_val is None:
        raise KeyError('DATE-OBS 계열 헤더 키워드를 찾을 수 없습니다. DATE_OBS_KEYS를 확인하세요.')

    if 'T' not in str(date_val):
        for key in TIME_OBS_KEYS:
            if key in header:
                date_val = f"{date_val}T{header[key]}"
                break
    return Time(date_val, format='isot', scale='utc')


def measure_flux(data, xy, gain=GAIN_DEFAULT, aperture_radius=APERTURE_RADIUS,
                  r_in=ANNULUS_R_IN, r_out=ANNULUS_R_OUT):
    """주어진 (x, y) 픽셀 좌표에서 배경을 뺀 구경측광 flux(ADU)와 오차를 계산.

    오차는 gain(e-/ADU)을 이용해 전자수 기준 광자잡음으로 환산한 뒤 ADU로 되돌린 근사값이다.
    """
    aperture = CircularAperture(xy, r=aperture_radius)
    annulus = CircularAnnulus(xy, r_in=r_in, r_out=r_out)

    ann_mask = annulus.to_mask(method='center')
    ann_data = ann_mask.multiply(data)
    ann_data_1d = ann_data[ann_mask.data > 0]
    bkg_mean, bkg_median, bkg_std = sigma_clipped_stats(ann_data_1d)

    phot_table = aperture_photometry(data, aperture)
    raw_sum = phot_table['aperture_sum'][0]
    n_pix = aperture.area
    net_flux = raw_sum - bkg_median * n_pix

    flux_err = np.sqrt(max(net_flux, 0) / gain + n_pix * bkg_std**2)
    return net_flux, flux_err, bkg_median


def refine_centroid(data, x0, y0, box_size=CENTROID_SEARCH_BOX):
    """예상 좌표 근처에서 실제 별 중심(centroid)을 다시 찾는다 (약간의 위치 흔들림 보정)."""
    try:
        x, y = centroid_sources(data, [x0], [y0], box_size=box_size,
                                  centroid_func=centroid_com)
        return float(x[0]), float(y[0])
    except Exception:
        return x0, y0


def instrumental_mag(flux, flux_err):
    if flux <= 0:
        return np.nan, np.nan
    mag = -2.5 * np.log10(flux)
    mag_err = 1.0857 * flux_err / flux
    return mag, mag_err


def check_star_list(target_xy, comp_xy_list):
    """대상/비교성 목록 점검: 대상 자신이 비교성에 들어 있으면 빼고, 바로 옆에 다른 별이 붙어 있으면 경고한다."""
    tx, ty = target_xy
    cleaned = []
    for xy in comp_xy_list:
        if np.hypot(xy[0] - tx, xy[1] - ty) < 3:
            print(f'[자동 수정] 비교성 목록에 대상 별 자신 {tuple(round(v, 1) for v in xy)}이 들어 있어 뺐습니다.')
            continue
        cleaned.append(xy)
    if 'sources' not in globals():
        return cleaned
    sx = np.array(sources[X_COL], dtype=float)
    sy = np.array(sources[Y_COL], dtype=float)
    sf = np.array(sources['flux'], dtype=float)
    warn_dist = ANNULUS_R_OUT + DETECT_FWHM
    for label, (x, y) in [('대상', target_xy)] + [(f'비교성 #{i}', xy) for i, xy in enumerate(cleaned)]:
        dist = np.hypot(sx - x, sy - y)
        self_flux = sf[np.argmin(dist)]
        near = (dist > 3) & (dist < warn_dist) & (sf > 0.05 * self_flux)
        if near.any():
            j = np.where(near)[0][np.argmin(dist[near])]
            advice = '구경 반지름을 줄이고, 결과 해석에 주의하세요.' if label == '대상' else '이 비교성은 빼는 것을 권장합니다.'
            print(f'[주의] {label} {(round(x, 1), round(y, 1))}: {dist[j]:.0f}px 거리에 이 별 밝기의 {sf[j] / self_flux:.0%}인 별이 '
                  f'붙어 있어 빛이 섞일 수 있습니다 (배경환 바깥 반지름 {ANNULUS_R_OUT}px). {advice}')
    return cleaned


# ---- 프레임 정렬: 사진마다 시야가 얼마나 이동/회전했는지 추정해서 측광 원을 별 위치로 옮깁니다 ----
from collections import Counter

ALIGN_CACHE = {}   # 파일 경로 -> 그 사진의 정렬 결과 (관측시각 jd, 변환 params, 오차 rms, 사용한 별 수 n)
_ALIGN = {}        # setup_alignment()가 채우는 기준 정보 (기준 별 좌표, 영상 중심)


def _com_centroid(data, x, y, half=12):
    """(x, y) 근처 (2*half+1)px 상자 안에서 별 중심을 무게중심으로 찾는다 (3번 반복해서 상자를 별 쪽으로 이동)."""
    h, w = data.shape
    for _ in range(3):
        x0, y0 = int(round(x)), int(round(y))
        if x0 - half < 0 or y0 - half < 0 or x0 + half + 1 > w or y0 + half + 1 > h:
            return None
        cut = data[y0 - half:y0 + half + 1, x0 - half:x0 + half + 1]
        border = np.concatenate([cut[0], cut[-1], cut[:, 0], cut[:, -1]])
        sub = cut - np.median(border)
        peak = sub.max()
        if peak <= 0:
            return None
        weight = np.clip(sub - 0.15 * peak, 0, None)
        if weight.sum() <= 0:
            return None
        yy, xx = np.mgrid[-half:half + 1, -half:half + 1]
        x = x0 + float((weight * xx).sum() / weight.sum())
        y = y0 + float((weight * yy).sum() / weight.sum())
    return x, y


def _predict(params, pts):
    """기준 사진의 좌표 pts(N,2)를 이 사진의 좌표로 옮긴다 (중심 기준 회전 + 평행이동)."""
    a, t = params
    c = _ALIGN['center']
    z = pts[:, 0] + 1j * pts[:, 1]
    zp = a * (z - c) + t + c
    return np.c_[zp.real, zp.imag]


def _fit_similarity(ref_pts, cur_pts):
    """기준 별 좌표 -> 이 사진의 별 좌표를 가장 잘 맞추는 (회전+배율 a, 평행이동 t)를 최소제곱으로 구한다."""
    c = _ALIGN['center']
    ref = ref_pts[:, 0] + 1j * ref_pts[:, 1] - c
    cur = cur_pts[:, 0] + 1j * cur_pts[:, 1] - c
    keep = np.ones(len(ref), bool)
    a, t, rms = 1 + 0j, 0j, np.inf
    for _ in range(3):
        design = np.c_[ref[keep], np.ones(int(keep.sum()))]
        a, t = np.linalg.lstsq(design, cur[keep], rcond=None)[0]
        res = np.abs(a * ref + t - cur)
        rms = float(np.sqrt(np.mean(res[keep] ** 2)))
        new_keep = res < max(1.0, 3.0 * rms)
        if new_keep.sum() < 4 or (new_keep == keep).all():
            break
        keep = new_keep
    return (a, t), rms, int(keep.sum())


def _estimate(data, guess, half=12):
    """guess로 예상한 기준 별 위치 근처에서 실제 별 중심을 찾아 정렬 변환을 구한다. 실패하면 None."""
    ref = _ALIGN['ref']
    pred = _predict(guess, ref)
    ok, cur = [], []
    for i, (x, y) in enumerate(pred):
        c = _com_centroid(data, x, y, half)
        if c is not None and np.hypot(c[0] - x, c[1] - y) < half - 2:
            ok.append(i)
            cur.append(c)
    if len(ok) < 4:
        return None
    params, rms, n = _fit_similarity(ref[ok], np.array(cur))
    if n < 4 or rms > 2.0:
        return None
    return {'params': params, 'rms': rms, 'n': n}


def _estimate_by_search(data):
    """가까운 사진의 값으로 못 찾을 때: 사진 전체에서 별을 새로 찾아 큰 이동(또는 180도 뒤집힘)을 추정한다."""
    _, med, std = sigma_clipped_stats(data[::4, ::4], sigma=3.0)
    found = DAOStarFinder(fwhm=DETECT_FWHM, threshold=DETECT_THRESHOLD_SIGMA * std)(data - med)
    if found is None or len(found) < 4:
        return None
    xcol = 'xcentroid' if 'xcentroid' in found.colnames else 'x_centroid'
    ycol = 'ycentroid' if 'ycentroid' in found.colnames else 'y_centroid'
    found.sort('flux')
    found.reverse()
    cur = np.array([[s[xcol], s[ycol]] for s in found[:30]])
    cur_z = cur[:, 0] + 1j * cur[:, 1]
    c = _ALIGN['center']
    ref_all = _ALIGN['ref_all']
    ref_z = ref_all[:, 0] + 1j * ref_all[:, 1]
    best = None
    for rot in (1 + 0j, -1 + 0j):
        moved = rot * (ref_z - c) + c
        diffs = (cur_z[None, :] - moved[:, None]).ravel()
        diffs = diffs[np.abs(diffs) < 300]
        if len(diffs) == 0:
            continue
        votes = Counter((int(round(d.real / 4)), int(round(d.imag / 4))) for d in diffs)
        (kx, ky), _ = votes.most_common(1)[0]
        near = diffs[np.abs(diffs - complex(kx * 4, ky * 4)) < 6]
        t = complex(np.median(near.real), np.median(near.imag))
        res = _estimate(data, (rot, t))
        if res is not None and (best is None or res['n'] > best['n']):
            best = res
    return best


def setup_alignment(ref_data, sources_table, num_stars=14):
    """기준 사진에서 정렬에 쓸 밝고 고립된 별들을 고른다. 쓸 수 있는 별 개수를 반환."""
    h, w = ref_data.shape
    pts = np.array([[float(s[X_COL]), float(s[Y_COL])] for s in sources_table[:max(num_stars * 3, 30)]])
    cand = pts[:num_stars * 2]
    chosen = []
    for min_gap in (50, 35, 25):
        chosen = []
        for i, (x, y) in enumerate(cand):
            if not (60 < x < w - 60 and 60 < y < h - 60):
                continue
            dist = np.hypot(pts[:, 0] - x, pts[:, 1] - y)
            if len(dist) > 1 and np.sort(dist)[1] < min_gap:
                continue
            chosen.append(i)
        if len(chosen) >= 4:
            break
    chosen = chosen[:num_stars]
    _ALIGN.clear()
    if len(chosen) < 4:
        return len(chosen)
    _ALIGN.update(center=complex(w / 2.0, h / 2.0), ref=cand[chosen], ref_all=pts[:14])
    return len(chosen)


def get_transform(path, data, header):
    """사진 한 장의 정렬 결과를 구해서 저장(캐시)하고 돌려준다. 관측시각이 가장 가까운 사진의 값을 출발점으로 쓴다."""
    if path in ALIGN_CACHE:
        return ALIGN_CACHE[path]
    try:
        jd = float(get_obs_time(header).jd)
    except Exception:
        jd = float(len(ALIGN_CACHE))
    guess = (1 + 0j, 0j)
    if ALIGN_CACHE:
        guess = min(ALIGN_CACHE.values(), key=lambda v: abs(v['jd'] - jd))['params']
    res = _estimate(data, guess)
    if res is None:
        res = _estimate_by_search(data)
    if res is not None:
        res = _estimate(data, res['params']) or res
    else:
        print(f'[경고] {os.path.basename(path)}: 별 위치 정렬에 실패해 가까운 사진의 값을 그대로 씁니다.')
        res = {'params': guess, 'rms': np.nan, 'n': 0}
    res['jd'] = jd
    ALIGN_CACHE[path] = res
    return res


def align_positions(path, data, header, xy_list):
    """기준 사진 기준의 좌표들(xy_list)이 이 사진에서는 어디에 있을지 계산해서 돌려준다 (정렬이 꺼져 있으면 그대로)."""
    if not ALIGN_ENABLED or not _ALIGN:
        return [tuple(xy) for xy in xy_list]
    res = get_transform(path, data, header)
    out = _predict(res['params'], np.array(xy_list, dtype=float).reshape(-1, 2))
    return [(float(x), float(y)) for x, y in out]


def align_info(path):
    res = ALIGN_CACHE.get(path)
    if res is None:
        return np.nan, np.nan
    return float(np.degrees(np.angle(res['params'][0]))), float(res['rms'])


print('함수 정의 완료')

## 5. 미리보기 & 별 자동 탐지

첫 번째 필터의 첫 번째 이미지를 보여주고, `DAOStarFinder`로 밝은 별들을 자동 탐지해서 번호를 매깁니다.
이 그림을 보고 **대상 별의 번호와 비교성(들)의 번호**를 다음 셀에서 지정하세요.

In [ ]:
PREVIEW_FILTER = FILTER_ORDER[0]  # 미리보기에 사용할 필터. 원하는 필터로 바꿔도 됨.

preview_file = list_fits_files(BASE_DIR, FILE_PATTERN_BY_FILTER[PREVIEW_FILTER])[0]
data, header = load_fits_data(preview_file)

mean, median, std = sigma_clipped_stats(data, sigma=3.0)
print(f'이미지: {preview_file}')
print(f'배경 median={median:.2f}, std={std:.2f}')

DETECT_FWHM = 8.0
DETECT_THRESHOLD_SIGMA = 8.0

finder = DAOStarFinder(fwhm=DETECT_FWHM, threshold=DETECT_THRESHOLD_SIGMA * std)
sources = finder(data - median)

if sources is None:
    raise RuntimeError(
        '별을 하나도 찾지 못했습니다. threshold가 너무 높거나(DETECT_THRESHOLD_SIGMA를 낮춰보세요, 예: 5.0) '
        'DETECT_FWHM이 실제 별 크기와 안 맞을 수 있습니다. 이 셀의 DETECT_FWHM/DETECT_THRESHOLD_SIGMA 값을 조정해서 다시 실행하세요.'
    )

# photutils 버전에 따라 컬럼명이 'xcentroid'/'ycentroid' 또는 'x_centroid'/'y_centroid'로 다를 수 있어 자동 감지
X_COL = 'xcentroid' if 'xcentroid' in sources.colnames else 'x_centroid'
Y_COL = 'ycentroid' if 'ycentroid' in sources.colnames else 'y_centroid'

print(f'탐지된 별 개수: {len(sources)}')
print(f'테이블 컬럼: {sources.colnames} (좌표 컬럼: {X_COL}, {Y_COL})')

sources.sort('flux')
sources.reverse()  # 밝은 순으로 정렬

zscale = ZScaleInterval()
vmin, vmax = zscale.get_limits(data)

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(data, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
for i, s in enumerate(sources[:40]):  # 너무 많으면 상위 40개만 표시
    ax.plot(s[X_COL], s[Y_COL], 'o', mfc='none', mec='red', ms=12)
    ax.text(s[X_COL] + 6, s[Y_COL] + 6, str(i), color='yellow', fontsize=9)
ax.set_title(f"{PREVIEW_FILTER} filter, first frame — detected stars (brightness rank)")
plt.show()

print(sources[['id', X_COL, Y_COL, 'flux']][:40])

## [설정] 대상 별 & 비교성 좌표 지정

위 미리보기 그림에서 확인한 번호를 아래에 입력하세요.
- 좌표를 직접 알고 있다면 `TARGET_XY`, `COMP_XY_LIST`에 픽셀 좌표 `(x, y)`를 바로 넣어도 됩니다.
- **`COMP_IDS`는 일단 `TARGET_TYPE`에 맞춰 통상적인 개수(변광성 3~5개 / 외계행성 식현상 5~10개)로 자동 채워집니다.** 미리보기 그림을 보고 실제로 밝고 고립된 별 번호로 바꿔서 다시 실행하면 됩니다 — 나중에 실전 데이터로 바꿔서 몇 번이고 다시 실행해도 됩니다.
- **주의: 번호 `0`은 "시야에서 가장 밝은 별"일 뿐, 관측 대상이라는 뜻이 아닙니다.** 대상이 어느 별인지는 별 차트(SIMBAD, Aladin 등)에서 대상의 좌표와 주변 별 배치를 확인한 뒤 번호를 정하세요. (쌍성처럼 가까운 별 쌍 중에서 **어두운 쪽**이 대상인 경우도 있습니다.)


In [ ]:
TARGET_ID = 0    # 위 그림/표에서 확인한 대상 별 번호

# 비교성 후보 번호 -- TARGET_TYPE에 따라 통상적인 개수를 기본으로 채워둡니다.
# 미리보기 그림을 보고 실제로 밝고 고립된(다른 별과 안 겹치는) 별 번호로 바꿔서 쓰세요.
if TARGET_TYPE == 1:
    COMP_IDS = [1, 2, 3, 4, 5]                  # 변광성: 3~5개 권장
else:
    COMP_IDS = [1, 2, 3, 4, 5, 6, 7, 8]         # 외계행성 식현상: 5~10개 권장

TARGET_XY = (float(sources[TARGET_ID][X_COL]), float(sources[TARGET_ID][Y_COL]))
COMP_XY_LIST = [(float(sources[i][X_COL]), float(sources[i][Y_COL])) for i in COMP_IDS]

print('대상 별 좌표:', TARGET_XY)
print('비교성 좌표들:', COMP_XY_LIST)
print(f'(TARGET_TYPE={TARGET_TYPE} 기준 기본 비교성 {len(COMP_IDS)}개 -- 미리보기 그림을 보고 COMP_IDS를 실제 좋은 별 번호로 바꾸면 됩니다)')

if TARGET_ID == 0:
    print('[주의] TARGET_ID=0은 "시야에서 가장 밝은 별"입니다. 정말 관측 대상이 이 별이 맞는지 별 차트와 대조해서 확인하세요.')

## 5.5 프레임 정렬 준비 (시야 이동/회전 추적)

밤새 관측하다 보면 망원경 추적이 완벽하지 않아서 **시야가 서서히 밀리거나(평행이동) 돌아갈(회전)** 수 있습니다. 측광 원을 처음 지정한 픽셀 좌표에 고정해두면, 별이 점점 원 밖으로 빠져나가면서 **별빛이 줄어든 것처럼 측정**됩니다 (실제로 별이 어두워진 게 아닌데도!).

특히 **시야 중심에서 멀리 떨어진 별일수록** 회전에 더 크게 밀립니다. (회전 1도 = 중심에서 1000픽셀 떨어진 별은 약 17픽셀 이동)

그래서 이 노트북은 사진마다 **밝고 고립된 별 여러 개의 위치를 기준 사진과 비교해서 "이동량 + 회전각"을 계산**하고, 대상/비교성의 측광 원을 그만큼 옮겨서 측정합니다. `ALIGN_ENABLED`로 켜고 끕니다 (기본값 켜짐).

- 수 도(°) 이내의 회전과 평행이동은 자동으로 따라갑니다. 망원경이 자오선을 넘으며 뒤집힌 경우(180도 회전)도 자동으로 시도합니다.
- 어느 사진에서 정렬에 실패하면 경고를 출력하고, 가까운 사진의 값을 대신 씁니다.


In [ ]:
if ALIGN_ENABLED:
    ref_img, _ = load_fits_data(preview_file)
    n_ref = setup_alignment(ref_img, sources, ALIGN_NUM_STARS)
    if n_ref >= 4:
        r_target = abs(complex(*TARGET_XY) - _ALIGN['center'])
        print(f'프레임 정렬 준비 완료: 기준 별 {n_ref}개 사용')
        print(f'대상 별은 시야 중심에서 {r_target:.0f}px 떨어져 있습니다 (시야가 1도 돌면 약 {r_target * np.pi / 180:.1f}px 밀립니다).')
        if r_target > 0.35 * min(ref_img.shape):
            print('[참고] 대상이 시야 가장자리에 가깝습니다. 다음 관측에서는 대상을 시야 중앙 쪽에 두면 회전 영향이 줄어듭니다.')
    else:
        print(f'[경고] 정렬에 쓸 수 있는 밝고 고립된 별이 {n_ref}개뿐이라 프레임 정렬을 건너뜁니다. (DETECT_THRESHOLD_SIGMA를 낮춰 별을 더 찾아보세요)')
else:
    print('ALIGN_ENABLED=False — 프레임 정렬을 건너뜁니다. (측광 원이 처음 지정한 픽셀 좌표에 고정됩니다)')


## (옵션) 비교성 안정성 검증

비교성을 3개 이상 넣으셨다면, 각 비교성이 실제로 안정적인지(그 자체가 변광성은 아닌지, 측광이 유난히 흔들리지는 않는지) 자동으로 확인해서 불안정한 비교성을 제외합니다.

**방법**: 관측 **전체 구간에서 고르게 뽑은** 사진들로, 각 비교성을 "나머지 비교성들의 중앙값 밝기" 대비 상대등급으로 측광해서 그 흩어짐(표준편차)을 계산합니다. 흩어짐이 `COMP_VET_MAX_STD`(mag)보다 크거나, 다른 비교성들보다 `COMP_VET_REL_FACTOR`배 이상 큰 별을 제외합니다. (앞쪽 사진만 보면 관측 도중에 변하는 별을 놓칠 수 있어서 전체 구간을 봅니다.)

- 비교성이 3개 미만이면 이 셀은 그냥 건너뜁니다.
- 이 셀을 실행하면 `COMP_XY_LIST`가 안정적인 별들만 남도록 갱신됩니다.

In [ ]:
COMP_VET_SAMPLE_SIZE = 25  # 관측 전체 구간에서 고르게 뽑을 사진 수 (앞쪽만 보면 나중에 변하는 별을 놓침)
COMP_VET_MAX_STD = 0.03    # 이 값(mag)보다 흩어짐이 크면 불안정한 비교성으로 판단해 제외
COMP_VET_REL_FACTOR = 3.0  # 다른 비교성들의 흩어짐(중앙값)보다 이 배수 이상 크면 제외 (최소 0.01 mag)


def vet_comparison_stars(files, comp_xy_list, max_std=COMP_VET_MAX_STD):
    if len(comp_xy_list) < 3:
        print('비교성이 3개 미만이라 안정성 검증을 건너뜁니다.')
        return comp_xy_list

    n = len(comp_xy_list)
    mag_series = [[] for _ in range(n)]
    for path in files:
        try:
            data, hdr = load_fits_data(path)
            fluxes = []
            for xy in align_positions(path, data, hdr, comp_xy_list):
                x, y = refine_centroid(data, *xy)
                f, _, _ = measure_flux(data, (x, y), gain=GAIN_DEFAULT,
                                         aperture_radius=APERTURE_RADIUS,
                                         r_in=ANNULUS_R_IN, r_out=ANNULUS_R_OUT)
                fluxes.append(f)
            if any(f <= 0 for f in fluxes):
                continue
            mags = [-2.5 * np.log10(f) for f in fluxes]
            for i in range(n):
                others_ref = np.median([mags[j] for j in range(n) if j != i])
                mag_series[i].append(mags[i] - others_ref)
        except Exception:
            continue

    stds = [np.std(s) if len(s) >= 3 else np.inf for s in mag_series]
    finite = [s for s in stds if np.isfinite(s)]
    limit = max_std
    if finite:
        limit = min(max_std, max(COMP_VET_REL_FACTOR * float(np.median(finite)), 0.01))
    print(f'비교성별 안정성 (나머지 비교성 중앙값 대비 상대등급의 std, 제외 기준 {limit:.4f} mag):')
    for i, (xy, s) in enumerate(zip(comp_xy_list, stds)):
        flag = '  -> 제외' if s > limit else ''
        print(f'  비교성 #{i} {tuple(round(v, 1) for v in xy)}: std={s:.4f}{flag}')

    kept = [xy for xy, s in zip(comp_xy_list, stds) if s <= limit]
    if len(kept) < 2:
        print('제외 기준을 적용하면 비교성이 2개 미만으로 남아, 원래 목록을 그대로 유지합니다.')
        return comp_xy_list
    return kept


_vet_all = list_fits_files(BASE_DIR, FILE_PATTERN_BY_FILTER[PREVIEW_FILTER])
_vet_idx = sorted(set(np.linspace(0, len(_vet_all) - 1, min(COMP_VET_SAMPLE_SIZE, len(_vet_all))).round().astype(int)))
vet_sample_files = [_vet_all[i] for i in _vet_idx]
COMP_XY_LIST = vet_comparison_stars(vet_sample_files, COMP_XY_LIST)
print('최종 비교성 좌표들:', COMP_XY_LIST)

## (교육) 비교성 개수에 따른 정밀도 변화

비교성을 몇 개 쓰느냐에 따라 측광 정밀도가 실제로 어떻게 달라지는지 직접 확인해봅니다. 비교성을 1개만 쓸 때 → 2개 → 3개 ··· 점점 늘려가면서, 대상 별의 차등 등급이 프레임마다 얼마나 흩어지는지(잡음, std)를 계산해서 그래프로 보여줍니다.

**원리**: 비교성 여러 개를 역분산 가중평균으로 합치면, 통계적으로 잡음이 대략 √N(N=비교성 개수)에 비례해서 줄어듭니다. 다만 그중 하나가 유난히 노이즈가 크거나 불안정하면 오히려 정밀도가 나빠질 수도 있습니다 — 그래서 이 그래프를 보면 "지금 고른 비교성 조합이 실제로 도움이 되고 있는지"를 눈으로 확인할 수 있습니다 (이 순서대로, 즉 `COMP_XY_LIST`에 넣은 순서대로 하나씩 추가합니다).

In [ ]:
PRECISION_TEST_SAMPLE_SIZE = 15  # 속도를 위해 샘플링할 최대 프레임 수


def precision_vs_num_comps(files, target_xy, comp_xy_list):
    """비교성을 1개씩 늘려가면서 대상 별 차등등급의 흩어짐(std)을 계산한다."""
    stds = []
    for k in range(1, len(comp_xy_list) + 1):
        subset = comp_xy_list[:k]
        diff_mags = []
        for path in files:
            try:
                data, hdr = load_fits_data(path)
                exp_xy = align_positions(path, data, hdr, [target_xy] + list(comp_xy_list))
                tx, ty = refine_centroid(data, *exp_xy[0])
                t_flux, t_err, _ = measure_flux(data, (tx, ty), gain=GAIN_DEFAULT,
                                                   aperture_radius=APERTURE_RADIUS,
                                                   r_in=ANNULUS_R_IN, r_out=ANNULUS_R_OUT)
                comp_mags, comp_errs = [], []
                for cx0, cy0 in exp_xy[1:1 + len(subset)]:
                    cx, cy = refine_centroid(data, cx0, cy0)
                    c_flux, c_err, _ = measure_flux(data, (cx, cy), gain=GAIN_DEFAULT,
                                                       aperture_radius=APERTURE_RADIUS,
                                                       r_in=ANNULUS_R_IN, r_out=ANNULUS_R_OUT)
                    c_mag, c_mag_err = instrumental_mag(c_flux, c_err)
                    if not np.isnan(c_mag) and c_mag_err > 0:
                        comp_mags.append(c_mag)
                        comp_errs.append(c_mag_err)
                if not comp_mags:
                    continue
                weights = 1.0 / np.array(comp_errs)**2
                c_mag = float(np.sum(np.array(comp_mags) * weights) / np.sum(weights))
                t_mag, _ = instrumental_mag(t_flux, t_err)
                if not np.isnan(t_mag) and not np.isnan(c_mag):
                    diff_mags.append(t_mag - c_mag)
            except Exception:
                continue
        stds.append(np.std(diff_mags) if len(diff_mags) >= 3 else np.nan)
    return stds


precision_sample_files = list_fits_files(BASE_DIR, FILE_PATTERN_BY_FILTER[PREVIEW_FILTER])[:PRECISION_TEST_SAMPLE_SIZE]
precision_stds = precision_vs_num_comps(precision_sample_files, TARGET_XY, COMP_XY_LIST)

n_comps = list(range(1, len(COMP_XY_LIST) + 1))
print('비교성 개수별 차등등급 흩어짐(std, mag) -- 작을수록 정밀함:')
for n, s in zip(n_comps, precision_stds):
    print(f'  비교성 {n}개: std={s:.4f}' if not np.isnan(s) else f'  비교성 {n}개: 측정 실패(프레임 부족)')

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(n_comps, precision_stds, 'o-')
ax.set_xlabel('Number of comparison stars used')
ax.set_ylabel('Differential magnitude scatter (std)')
ax.set_title('Precision vs Number of Comparison Stars')
ax.set_xticks(n_comps)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## (옵션) 구경 반지름 자동 최적화

비교성 2개의 상대 밝기 차이(comp1 − comp2)는 둘 다 안정적인 별이라는 가정 하에 시간이 지나도 거의 일정해야 합니다. 이 차이의 흩어짐(표준편차)이 **가장 작아지는 구경 반지름**을 여러 후보 중에서 찾아 `APERTURE_RADIUS`/`ANNULUS_R_IN`/`ANNULUS_R_OUT`를 자동으로 갱신합니다.

- 비교성이 최소 2개 있어야 합니다.
- 속도를 위해 프레임 전체가 아니라 앞쪽 일부만 샘플링해서 계산합니다.

In [ ]:
RADII_TO_TEST = list(range(6, 23, 2))  # 시도해볼 구경 반지름 후보 (px)
APERTURE_SAMPLE_SIZE = 15               # 속도를 위해 샘플링할 최대 프레임 수
ANNULUS_GAP = 4                         # 최적 반지름 + 이 값 = ANNULUS_R_IN
ANNULUS_WIDTH = 6                       # ANNULUS_R_IN + 이 값 = ANNULUS_R_OUT


def optimize_aperture_radius(files, comp_xy_list, radii):
    """비교성 두 개의 상대밝기(comp1-comp2) 흩어짐이 최소가 되는 구경 반지름을 찾는다."""
    if len(comp_xy_list) < 2:
        raise ValueError('비교성이 2개 이상 있어야 구경 최적화를 할 수 있습니다.')
    c1_xy, c2_xy = comp_xy_list[0], comp_xy_list[1]

    stds = []
    for r in radii:
        r_in, r_out = r + ANNULUS_GAP, r + ANNULUS_GAP + ANNULUS_WIDTH
        diffs = []
        for path in files:
            try:
                data, hdr = load_fits_data(path)
                e1, e2 = align_positions(path, data, hdr, [c1_xy, c2_xy])
                x1, y1 = refine_centroid(data, *e1)
                x2, y2 = refine_centroid(data, *e2)
                f1, _, _ = measure_flux(data, (x1, y1), gain=GAIN_DEFAULT,
                                          aperture_radius=r, r_in=r_in, r_out=r_out)
                f2, _, _ = measure_flux(data, (x2, y2), gain=GAIN_DEFAULT,
                                          aperture_radius=r, r_in=r_in, r_out=r_out)
                if f1 > 0 and f2 > 0:
                    diffs.append((-2.5 * np.log10(f1)) - (-2.5 * np.log10(f2)))
            except Exception:
                continue
        stds.append(np.std(diffs) if len(diffs) >= 3 else np.nan)
    return stds


aperture_sample_files = list_fits_files(BASE_DIR, FILE_PATTERN_BY_FILTER[PREVIEW_FILTER])[:APERTURE_SAMPLE_SIZE]
scatter_by_radius = optimize_aperture_radius(aperture_sample_files, COMP_XY_LIST, RADII_TO_TEST)

valid = [(r, s) for r, s in zip(RADII_TO_TEST, scatter_by_radius) if not np.isnan(s)]
if not valid:
    print('구경 최적화에 실패했습니다 (측광 가능한 프레임이 부족합니다). 기존 값을 유지합니다.')
else:
    best_radius, best_std = min(valid, key=lambda t: t[1])
    print('반지름별 비교성 상대밝기 흩어짐(std, mag):')
    for r, s in valid:
        marker = '  <- 최적' if r == best_radius else ''
        print(f'  r={r:2d}px  std={s:.4f}{marker}')

    fig, ax = plt.subplots(figsize=(7, 4))
    rs, ss = zip(*valid)
    ax.plot(rs, ss, 'o-')
    ax.axvline(best_radius, color='r', ls='--', label=f'최적 반지름 = {best_radius}px')
    ax.set_xlabel('Aperture radius (px)')
    ax.set_ylabel('Comp1-Comp2 magnitude scatter (std)')
    ax.set_title('Aperture Radius Optimization')
    ax.legend()
    plt.tight_layout()
    plt.show()

    APERTURE_RADIUS = best_radius
    ANNULUS_R_IN = best_radius + ANNULUS_GAP
    ANNULUS_R_OUT = best_radius + ANNULUS_GAP + ANNULUS_WIDTH
    print(f'APERTURE_RADIUS={APERTURE_RADIUS}, ANNULUS_R_IN={ANNULUS_R_IN}, ANNULUS_R_OUT={ANNULUS_R_OUT} 로 갱신했습니다.')

## 6. 필터별 시계열 측광 실행

각 필터(`FILE_PATTERN_BY_FILTER`)의 모든 FITS 파일에 대해:
1. 관측 시각(및 있으면 `AIRMASS`, 6.6절 추세 제거에 사용) 추출
2. 대상/비교성 위치를 이미지별로 재탐색 (`ALIGN_ENABLED`가 켜져 있으면 시야 이동/회전을 반영한 위치 근처에서 찾음) — 정렬로 예상한 좌표에서 얼마나 벗어났는지(`target_shift_px`, `comp_shift_px_max`)도 같이 기록 (6.5절 아웃라이어 제거에 사용)
3. 구경측광 → 기기 등급 계산
4. 비교성들을 **역분산 가중 평균**으로 결합해 하나의 앙상블 비교 등급으로 만들고 차등 등급 계산

In [ ]:
def process_filter(filter_name, files, target_xy, comp_xy_list):
    rows = []
    for path in files:
        try:
            data, header = load_fits_data(path)
            t_obs = get_obs_time(header)
            airmass = header.get('AIRMASS', np.nan)

            exp_xy = align_positions(path, data, header, [target_xy] + list(comp_xy_list))
            target_exp = exp_xy[0]
            tx, ty = refine_centroid(data, *target_exp)
            target_shift_px = float(np.hypot(tx - target_exp[0], ty - target_exp[1]))
            t_flux, t_err, _ = measure_flux(data, (tx, ty), gain=GAIN_DEFAULT,
                                              aperture_radius=APERTURE_RADIUS,
                                              r_in=ANNULUS_R_IN, r_out=ANNULUS_R_OUT)

            comp_mags, comp_mag_errs, comp_shifts = [], [], []
            comp_cols = {}
            for k_c, (cx0, cy0) in enumerate(exp_xy[1:]):
                cx, cy = refine_centroid(data, cx0, cy0)
                comp_shifts.append(float(np.hypot(cx - cx0, cy - cy0)))
                c_flux, c_err, _ = measure_flux(data, (cx, cy), gain=GAIN_DEFAULT,
                                                  aperture_radius=APERTURE_RADIUS,
                                                  r_in=ANNULUS_R_IN, r_out=ANNULUS_R_OUT)
                c_mag, c_mag_err = instrumental_mag(c_flux, c_err)
                comp_cols[f'comp{k_c}_mag'] = c_mag
                comp_cols[f'comp{k_c}_err'] = c_mag_err
                if not np.isnan(c_mag) and c_mag_err > 0:
                    comp_mags.append(c_mag)
                    comp_mag_errs.append(c_mag_err)
            comp_shift_px_max = max(comp_shifts) if comp_shifts else np.nan

            if not comp_mags:
                raise ValueError('모든 비교성 측광에 실패했습니다.')

            weights = 1.0 / np.array(comp_mag_errs)**2
            c_mag = float(np.sum(np.array(comp_mags) * weights) / np.sum(weights))
            c_mag_err = float(np.sqrt(1.0 / np.sum(weights)))

            t_mag, t_mag_err = instrumental_mag(t_flux, t_err)

            diff_mag = t_mag - c_mag
            diff_mag_err = np.sqrt(t_mag_err**2 + c_mag_err**2)

            rows.append({
                'filter': filter_name,
                'file': os.path.basename(path),
                'jd': t_obs.jd,
                'datetime': t_obs.isot,
                'airmass': airmass,
                'target_flux': t_flux,
                'target_instmag': t_mag,
                'diff_mag': diff_mag,
                'diff_mag_err': diff_mag_err,
                'target_shift_px': target_shift_px,
                'comp_shift_px_max': comp_shift_px_max,
                'align_rot_deg': align_info(path)[0],
                'align_rms_px': align_info(path)[1],
                'exptime': header.get('EXPTIME', header.get('EXPOSURE', np.nan)),
                **comp_cols,
            })
        except Exception as e:
            print(f"[경고] {path} 처리 실패: {e}")
    return pd.DataFrame(rows).sort_values('jd').reset_index(drop=True)


COMP_XY_LIST = check_star_list(TARGET_XY, COMP_XY_LIST)

results = {}
for fname, pattern in FILE_PATTERN_BY_FILTER.items():
    try:
        files = list_fits_files(BASE_DIR, pattern)
    except FileNotFoundError:
        print(f'--- {fname} 필터: \'{pattern}\' 파일이 없어 건너뜁니다 ---')
        continue
    print(f'--- {fname} 필터 처리 중 ({len(files)}장) ---')
    results[fname] = process_filter(fname, files, TARGET_XY, COMP_XY_LIST)
    print(f'{fname}: {len(results[fname])}장 처리 완료')

if not results:
    raise FileNotFoundError('어떤 필터의 파일도 찾지 못했습니다. BASE_DIR와 FILE_PATTERN_BY_FILTER를 확인하세요.')

FILTER_ORDER = [f for f in FILTER_ORDER if f in results]
print('분석된 필터:', FILTER_ORDER)

all_df = pd.concat(results.values(), ignore_index=True)
all_df.head()

## 6.1 프레임 정렬 요약

밤새 시야가 얼마나 이동/회전했는지 보여줍니다. 회전이 크면(예: 1도 이상) **정렬 없이는 가장자리 별이 측광 원을 벗어났을** 것입니다.


In [ ]:
if ALIGN_ENABLED and _ALIGN and ALIGN_CACHE:
    items = sorted(ALIGN_CACHE.values(), key=lambda v: v['jd'])
    t_h = np.array([(v['jd'] - items[0]['jd']) * 24.0 for v in items])
    rot_deg = np.array([np.degrees(np.angle(v['params'][0])) for v in items])
    rot_deg = rot_deg - rot_deg[0]
    shift_px = np.array([abs(v['params'][1]) for v in items])
    n_fail = sum(1 for v in items if v['n'] == 0)

    target_move = []
    for v in items:
        px, py = _predict(v['params'], np.array([TARGET_XY]))[0]
        target_move.append(float(np.hypot(px - TARGET_XY[0], py - TARGET_XY[1])))
    print(f'정렬한 사진: {len(items)}장 (실패 {n_fail}장), 정렬 오차 중앙값 {np.nanmedian([v["rms"] for v in items]):.2f}px')
    print(f'시야 회전: 처음 사진 대비 {rot_deg.min():+.2f} ~ {rot_deg.max():+.2f}도, 평행이동 최대 {shift_px.max():.1f}px')
    print(f'대상 별이 처음 지정한 좌표에서 최대 {max(target_move):.1f}px 이동 (구경 반지름 {APERTURE_RADIUS}px)')
    if max(target_move) > APERTURE_RADIUS / 2:
        print('-> 정렬 없이 측광 원을 고정했다면 대상 별이 원 밖으로 빠져 밝기가 잘못 측정됐을 정도의 이동입니다.')

    fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
    axes[0].plot(t_h, rot_deg, 'k.-')
    axes[0].set_ylabel('Field rotation (deg)')
    axes[0].grid(alpha=0.3)
    axes[1].plot(t_h, shift_px, 'b.-')
    axes[1].set_ylabel('Field shift at center (px)')
    axes[1].set_xlabel('Hours since first frame')
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('프레임 정렬을 사용하지 않았거나 정렬 결과가 없어 요약을 건너뜁니다.')


## 6.5 아웃라이어 프레임 자동 제거

측광은 됐지만 신뢰할 수 없는 프레임(추적이 엉뚱한 별로 튄 경우, 유난히 노이즈가 큰 경우)을 제거합니다. **광도값 자체를 임의로 잘라내는 게 아니라**, 독립적인 품질 지표 두 가지로 판단합니다:

- `target_shift_px` / `comp_shift_px_max`: 대상/비교성 위치가 지정한 좌표에서 `MAX_CENTROID_SHIFT_PX`(px) 이상 벗어난 프레임
- `diff_mag_err`: 그 필터의 오차 중앙값 × `MAX_DIFF_MAG_ERR_MULTIPLIER`(절대 하한 `MIN_DIFF_MAG_ERR_FLOOR`)를 넘는 저품질 프레임

어떤 파일이 왜 제외됐는지 전부 출력하므로, 필요하면 설정 셀의 기준값을 조정해서 다시 실행하세요.

In [ ]:
for fname in FILTER_ORDER:
    df = results.get(fname)
    if df is None or df.empty:
        continue

    median_err = df['diff_mag_err'].median()
    err_threshold = max(median_err * MAX_DIFF_MAG_ERR_MULTIPLIER, MIN_DIFF_MAG_ERR_FLOOR)

    bad_shift = df['target_shift_px'] > MAX_CENTROID_SHIFT_PX
    bad_comp_shift = df['comp_shift_px_max'] > MAX_CENTROID_SHIFT_PX
    bad_err = df['diff_mag_err'] > err_threshold
    bad = bad_shift | bad_comp_shift | bad_err

    n_bad = int(bad.sum())
    if n_bad > 0:
        print(f'--- {fname} 필터: {n_bad}/{len(df)}장 제외 (오차 임계값 {err_threshold:.4f}mag) ---')
        for _, row in df[bad].iterrows():
            reasons = []
            if row['target_shift_px'] > MAX_CENTROID_SHIFT_PX:
                reasons.append(f"대상 shift={row['target_shift_px']:.1f}px")
            if row['comp_shift_px_max'] > MAX_CENTROID_SHIFT_PX:
                reasons.append(f"비교성 shift={row['comp_shift_px_max']:.1f}px")
            if row['diff_mag_err'] > err_threshold:
                reasons.append(f"오차={row['diff_mag_err']:.4f}mag")
            print(f"  {row['file']}: {', '.join(reasons)}")
    else:
        print(f'--- {fname} 필터: 제외된 프레임 없음 ({len(df)}장 모두 유지) ---')

    results[fname] = df[~bad].reset_index(drop=True)

RESULTS_UNBINNED = {k: v.copy() for k, v in results.items()}  # 추세 제거/비닝 전 값 (11절 식 곡선 피팅에 사용)

all_df = pd.concat(results.values(), ignore_index=True)
print(f'\n필터링 후 총 {len(all_df)}장')
all_df.head()

## 6.6 (옵션) 추세 제거 (Detrending)

대기 소광(밤 동안 대상의 고도가 바뀌면서 생기는 밝기 변화)은 차등측광으로 상당 부분 상쇄되지만, 대상과 비교성의 색(스펙트럼)이 다르면 소광되는 정도도 미묘하게 달라서(2차 소광) 완만한 추세가 남을 수 있습니다.

**방법**: 헤더에 `AIRMASS`가 있으면 그 값에 대해, 없으면 경과 시간(시간 단위)에 대해 `diff_mag`를 `DETREND_DEGREE`차 다항식으로 피팅한 뒤 그 추세만 빼줍니다. 원본 값은 `diff_mag_raw` 컬럼에 그대로 남습니다.

**주의**: 대상 자체가 관측 시간 내내 크게 변한다면 이 보정이 실제 신호까지 같이 깎아버릴 수 있습니다. 그래서 `DETRENDING_ENABLED`는 설정 셀의 **`TARGET_TYPE`(1=변광성 → 꺼짐, 2=외계행성 식현상 → 켜짐)** 으로 자동 결정됩니다. 직접 반대로 켜고 끈 결과를 비교해보고 싶으면 설정 셀 맨 아래에서 덮어쓰세요.

In [ ]:
def detrend_series(df, degree=DETREND_DEGREE):
    if 'airmass' in df.columns and df['airmass'].notna().sum() >= max(degree + 2, 5):
        x = df['airmass'].values.astype(float)
        x_label = 'AIRMASS'
    else:
        x = (df['jd'].values - df['jd'].values.min()) * 24.0
        x_label = '경과 시간(시간, AIRMASS 헤더 없음)'

    coeffs = np.polyfit(x, df['diff_mag'].values, deg=degree)
    trend = np.polyval(coeffs, x)
    detrended = df['diff_mag'].values - trend + np.mean(trend)
    return detrended, x_label


if DETRENDING_ENABLED:
    for fname in FILTER_ORDER:
        df = results.get(fname)
        if df is None or df.empty or len(df) < DETREND_DEGREE + 2:
            continue
        detrended, x_label = detrend_series(df)
        df = df.copy()
        df['diff_mag_raw'] = df['diff_mag']
        df['diff_mag'] = detrended
        results[fname] = df
        print(f'{fname} 필터: {x_label} 기준 {DETREND_DEGREE}차 추세 제거 완료 (원본은 diff_mag_raw에 보존)')
    all_df = pd.concat(results.values(), ignore_index=True)
else:
    print('DETRENDING_ENABLED=False — 추세 제거를 건너뜁니다. (설정 셀에서 켤 수 있습니다)')

## 6.7 (옵션) 시간 구간 비닝 (Binning)

가까운 시간(`BIN_WIDTH_MINUTES`분 이내)에 찍힌 프레임들을 하나로 묶어서 평균을 냅니다. 점 개수는 줄지만 점 하나의 정밀도는 올라갑니다.

`BINNING_ENABLED`도 설정 셀의 **`TARGET_TYPE`**(1=변광성 → 꺼짐, 2=외계행성 식현상 → 켜짐)으로 자동 결정됩니다 — 변광성은 시간 해상도가 중요해서 기본 꺼짐, 트랜싯은 점당 정밀도가 더 중요해서 기본 켜짐입니다.

In [ ]:
def bin_by_time(df, bin_width_minutes=BIN_WIDTH_MINUTES):
    if df.empty:
        return df
    bin_width_days = bin_width_minutes / (24 * 60)
    bin_id = np.floor((df['jd'].values - df['jd'].values.min()) / bin_width_days).astype(int)
    rows = []
    for b in np.unique(bin_id):
        group = df[bin_id == b]
        weights = 1.0 / group['diff_mag_err'].values**2
        mag = float(np.sum(group['diff_mag'].values * weights) / np.sum(weights))
        err = float(np.sqrt(1.0 / np.sum(weights)))
        rows.append({
            'filter': group['filter'].iloc[0],
            'jd': float(group['jd'].mean()),
            'diff_mag': mag,
            'diff_mag_err': err,
            'n_frames': len(group),
        })
    return pd.DataFrame(rows).sort_values('jd').reset_index(drop=True)


if BINNING_ENABLED:
    for fname in FILTER_ORDER:
        df = results.get(fname)
        if df is None or df.empty:
            continue
        binned = bin_by_time(df)
        print(f'{fname} 필터: {len(df)}장 -> {len(binned)}개 구간 ({BIN_WIDTH_MINUTES}분 단위)으로 비닝')
        results[fname] = binned
    all_df = pd.concat(results.values(), ignore_index=True)
else:
    print('BINNING_ENABLED=False — 비닝을 건너뜁니다. (설정 셀에서 켤 수 있습니다)')

## 7. 결과 저장 (CSV)

In [ ]:
OUTPUT_CSV = r'C:\26astro\YZBootis\result\photometry_results.csv'  # 실제 저장하고 싶은 로컬 경로로 수정

os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)  # 폴더가 없으면 자동 생성
all_df.to_csv(OUTPUT_CSV, index=False)
print(f'저장 완료: {OUTPUT_CSV}')

## 8. 필터별 광도곡선 그래프

In [ ]:
cmap = plt.cm.tab10
colors = {f: cmap(i % 10) for i, f in enumerate(FILTER_ORDER)}

fig, axes = plt.subplots(len(results), 1, figsize=(10, 4 * len(results)), sharex=True)
if len(results) == 1:
    axes = [axes]

t0 = all_df['jd'].min()

for ax, fname in zip(axes, FILTER_ORDER):
    df = results.get(fname)
    if df is None or df.empty:
        continue
    hours = (df['jd'] - t0) * 24.0
    ax.errorbar(hours, df['diff_mag'], yerr=df['diff_mag_err'],
                fmt='o', ms=4, capsize=2, color=colors.get(fname, 'k'), label=f'{fname} filter')
    ax.invert_yaxis()  # 등급은 작을수록 밝음
    ax.set_ylabel('Differential Magnitude')
    ax.legend(loc='best')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel(f'Time (hours since JD {t0:.5f})')
fig.suptitle(f"Filter-by-Filter Light Curve (Differential Photometry) — {'/'.join(FILTER_ORDER)}")
plt.tight_layout()
plt.show()

## 9. 통합 그래프 (한 화면에 모든 필터 비교)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
for fname in FILTER_ORDER:
    df = results.get(fname)
    if df is None or df.empty:
        continue
    hours = (df['jd'] - t0) * 24.0
    ax.errorbar(hours, df['diff_mag'], yerr=df['diff_mag_err'],
                fmt='o-', ms=4, capsize=2, color=colors.get(fname, 'k'),
                label=f'{fname} filter', alpha=0.8)

ax.invert_yaxis()
ax.set_xlabel(f'Time (hours since JD {t0:.5f})')
ax.set_ylabel('Differential Magnitude')
ax.set_title(f"Combined {'/'.join(FILTER_ORDER)} Light Curve")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. (옵션) 색지수 시계열

**옵션 기능입니다**: 설정 셀의 `COLOR_INDEX_ENABLED`로 켜고 끕니다. 필터를 1개만 분석했다면 자동으로 건너뜁니다.

서로 다른 필터로 거의 같은 시각에 찍은 이미지가 있을 때, 시간이 가까운 관측끼리 짝지어 색지수의 시간 변화를 볼 수 있습니다. `FILTER_ORDER`에 넣은 순서대로 **인접한 필터끼리 자동으로 짝지어** 계산합니다. 필터 전환 사이 시간 간격이 너무 크면 의미가 없어지므로 `TOLERANCE_MIN`(분 단위 허용 오차)을 데이터 촬영 주기에 맞게 조정하세요.

In [ ]:
TOLERANCE_MIN = 5.0  # 필터 간 매칭을 허용할 최대 시간 차이 (분)

def merge_by_time(df_a, df_b, label_a, label_b, tol_min=TOLERANCE_MIN):
    a = df_a[['jd', 'diff_mag']].rename(columns={'diff_mag': f'mag_{label_a}'}).sort_values('jd')
    b = df_b[['jd', 'diff_mag']].rename(columns={'diff_mag': f'mag_{label_b}'}).sort_values('jd')
    merged = pd.merge_asof(a, b, on='jd', direction='nearest',
                            tolerance=tol_min / (24 * 60))
    return merged.dropna()

if not COLOR_INDEX_ENABLED:
    print('COLOR_INDEX_ENABLED=False — 색지수 그래프를 건너뜁니다. (설정 셀에서 켤 수 있습니다)')
elif len(FILTER_ORDER) < 2:
    print('분석한 필터가 1개뿐이라 색지수를 계산할 수 없어 건너뜁니다.')
else:
    fig, ax = plt.subplots(figsize=(11, 5))

    for i, (f1, f2) in enumerate(zip(FILTER_ORDER, FILTER_ORDER[1:])):
        df1, df2 = results.get(f1), results.get(f2)
        if df1 is None or df2 is None or df1.empty or df2.empty:
            continue
        merged = merge_by_time(df1, df2, f1, f2)
        if merged.empty:
            continue
        label = f'{f1}-{f2}'
        merged[label] = merged[f'mag_{f1}'] - merged[f'mag_{f2}']
        ax.plot((merged['jd'] - t0) * 24.0, merged[label], 'o-', label=label, color=cmap((i + 3) % 10))

    ax.set_xlabel(f'Time (hours since JD {t0:.5f})')
    ax.set_ylabel('Color Index')
    ax.set_title('Color Index vs Time')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

## 11. (식현상) 식 곡선 피팅 — 추세와 식 깊이를 동시에 맞추기

외계행성 식은 몇 시간에 걸친 얕은 감광(보통 0.5~2%)이라, 대기질량 변화 같은 완만한 추세와 섞이기 쉽습니다. 이 셀은 **"추세(직선) + 식 모양(사다리꼴)"을 동시에 데이터에 맞춰서** 식 깊이를 숫자(± 오차)로 알려주고 그래프로 보여줍니다.

- 식 시각은 행성의 **궤도 정보(기준 식 시각 `TRANSIT_T0_BJD`, 주기 `TRANSIT_PERIOD_D`)**로 예보해서 씁니다. 기본값은 HAT-P-1b 예시이니, 다른 행성이면 문헌이나 식 예보 사이트(ETD, ExoClock 등)의 값으로 바꾸세요.
- 6.6절 추세 제거나 6.7절 비닝을 켰든 껐든, 이 셀은 **그 전 단계의 원본 값**(6.5절 아웃라이어 제거 직후)으로 직접 피팅합니다.
- 오차는 10분 묶음의 실제 흩어짐으로 계산해서, 구름이나 시상 변화 같은 느린 잡음까지 반영합니다 (사진 한 장의 오차막대보다 훨씬 현실적입니다).
- **비교성을 하나씩 빼고 다시 맞춰봅니다.** 결과가 비교성 구성에 따라 크게 바뀌면 진짜 식이 아니라 비교성 쪽 잡음일 가능성이 큽니다.
- 식 전·후의 평평한 구간이 짧으면 추세와 식을 구분하기 어려워 깊이가 부정확해집니다. 이 경우 셀이 경고를 출력합니다.

**결과 읽는 법**: 깊이가 오차의 3배 이상(3시그마)이고, 비교성을 하나씩 빼도 비슷한 값이 나오면 믿을 만한 검출입니다. 반대로 비교성 하나에 따라 값이 크게 바뀐다면, 원하는 값이 나오는 조합을 고르지 말고 "이 데이터로는 판단하기 어렵다"고 보는 것이 맞습니다.


In [ ]:
# ==== [설정] 식 곡선 피팅 ====
FIT_ENABLED = (TARGET_TYPE == 2)     # 외계행성 식현상(TARGET_TYPE=2)이면 기본으로 켜짐
FIT_FILTER = FILTER_ORDER[0]         # 피팅할 필터

# 행성 궤도 정보 — 아래는 HAT-P-1b 예시입니다. 다른 행성이면 문헌/식 예보 사이트 값으로 바꾸세요.
TRANSIT_T0_BJD = 2456288.492023      # 기준 식 중앙 시각 (BJD_TDB)
TRANSIT_PERIOD_D = 4.46529962        # 공전 주기 (일)
TRANSIT_DURATION_H = 2.798           # 식 전체 길이 (시작~끝, 시간)
TRANSIT_INGRESS_H = 0.51             # 식에 완전히 들어가는(나오는) 데 걸리는 시간
EXPECTED_DEPTH_MMAG = 15.0           # 문헌상 예상 깊이 (그래프 비교용 점선)
TARGET_RA_DEC = ('22h57m46.84s', '+38d40m29.8s')   # 대상 별 적경/적위 (BJD 시각 변환용)

FIT_TREND = 'airmass'     # 'airmass'(대기질량), 'time'(시간), 'none'(추세 없음)
FIT_FREE_TIMING = False   # True면 식 중앙 시각도 예보 ±FIT_TIMING_WINDOW_MIN분 안에서 가장 잘 맞는 값으로 찾음
FIT_TIMING_WINDOW_MIN = 60
FIT_BIN_MINUTES = 10.0    # 오차 계산과 피팅에 쓰는 묶음 시간(분)

from astropy.coordinates import SkyCoord, EarthLocation
import astropy.units as u


def to_bjd_tdb(jd_utc, ra_dec):
    """관측 시각(UTC JD)을 태양계 질량중심 기준 시각(BJD_TDB)으로 바꾼다. 궤도 정보가 BJD 기준이라 필요."""
    coord = SkyCoord(ra_dec[0], ra_dec[1])
    geocenter = EarthLocation.from_geocentric(0 * u.m, 0 * u.m, 0 * u.m)
    t = Time(jd_utc, format='jd', scale='utc', location=geocenter)
    return (t.tdb + t.light_travel_time(coord, 'barycentric')).jd


def contact_times(tc):
    dur, ing = TRANSIT_DURATION_H / 24, TRANSIT_INGRESS_H / 24
    return tc - dur / 2, tc - dur / 2 + ing, tc + dur / 2 - ing, tc + dur / 2


def transit_shape(bjd, tc):
    """식 모양(사다리꼴): 식 밖 0, 식 바닥 1, 들어가고 나오는 구간은 직선으로 연결."""
    t1, t2, t3, t4 = contact_times(tc)
    ing = t2 - t1
    s = np.zeros_like(bjd, dtype=float)
    s[(bjd >= t2) & (bjd <= t3)] = 1.0
    m = (bjd > t1) & (bjd < t2)
    s[m] = (bjd[m] - t1) / ing
    m = (bjd > t3) & (bjd < t4)
    s[m] = (t4 - bjd[m]) / ing
    return s


def fit_transit(bjd, d, trend, tc):
    """묶음 데이터에 '상수 + 추세 + 깊이 x 식모양'을 최소제곱으로 맞춘다.
    반환: (계수, 깊이 오차, 잔차 rms, 식 없는 모델의 잔차 rms)"""
    bid = np.floor((bjd - bjd.min()) / (FIT_BIN_MINUTES / 1440)).astype(int)
    groups = [b for b in np.unique(bid) if (bid == b).sum() >= 3]

    def binned(v):
        return np.array([np.nanmean(v[bid == b]) for b in groups])

    y = binned(d)
    base = [np.ones(len(groups))] + ([binned(trend)] if trend is not None else [])
    A = np.column_stack(base + [binned(transit_shape(bjd, tc))])
    ok = np.isfinite(y) & np.all(np.isfinite(A), axis=1)
    y, A = y[ok], A[ok]
    coef = np.linalg.lstsq(A, y, rcond=None)[0]
    res = y - A @ coef
    dof = max(len(y) - A.shape[1], 1)
    cov = (np.sum(res ** 2) / dof) * np.linalg.pinv(A.T @ A)
    A0 = A[:, :-1]
    res0 = y - A0 @ np.linalg.lstsq(A0, y, rcond=None)[0]
    return coef, float(np.sqrt(cov[-1, -1])), float(np.sqrt(np.mean(res ** 2))), float(np.sqrt(np.mean(res0 ** 2)))


TRANSIT_FIT = None
fit_source = globals().get('RESULTS_UNBINNED', results)
fdf = fit_source.get(FIT_FILTER) if FIT_ENABLED else None

if not FIT_ENABLED:
    print('FIT_ENABLED=False — 식 곡선 피팅을 건너뜁니다. (외계행성 식현상이면 TARGET_TYPE=2로 두거나 FIT_ENABLED=True)')
elif fdf is None or fdf.empty or 'n_frames' in fdf.columns:
    print('피팅할 원본(비닝 전) 데이터가 없습니다. 6번 측광부터 6.5절 아웃라이어 제거 셀까지 다시 실행한 뒤 이 셀을 실행하세요.')
else:
    fdf = fdf.sort_values('jd').reset_index(drop=True)
    exp_s = fdf['exptime'].values.astype(float) if 'exptime' in fdf.columns else np.full(len(fdf), np.nan)
    jd_mid = fdf['jd'].values + np.where(np.isfinite(exp_s), exp_s, 0.0) / 2 / 86400
    bjd = to_bjd_tdb(jd_mid, TARGET_RA_DEC)
    d = fdf['diff_mag'].values.astype(float)
    hours = (bjd - bjd[0]) * 24

    airmass_ok = 'airmass' in fdf.columns and np.isfinite(fdf['airmass'].astype(float)).mean() > 0.9
    if FIT_TREND == 'none':
        trend, trend_name = None, '없음'
    elif FIT_TREND == 'airmass' and airmass_ok:
        trend = fdf['airmass'].astype(float).interpolate(limit_direction='both').values
        trend_name = '대기질량'
    else:
        trend, trend_name = hours, '시간'

    epoch = np.round((np.median(bjd) - TRANSIT_T0_BJD) / TRANSIT_PERIOD_D)
    tc = TRANSIT_T0_BJD + epoch * TRANSIT_PERIOD_D
    if FIT_FREE_TIMING:
        offsets = np.arange(-FIT_TIMING_WINDOW_MIN, FIT_TIMING_WINDOW_MIN + 1, 2) / 1440
        scores = [fit_transit(bjd, d, trend, tc + o)[2] if transit_shape(bjd, tc + o).max() > 0 else np.inf for o in offsets]
        tc = tc + offsets[int(np.argmin(scores))]

    t1, t2, t3, t4 = contact_times(tc)
    utc_offset = bjd[0] - jd_mid[0]

    def utc_str(b):
        return Time(b - utc_offset, format='jd', scale='utc').iso[11:16]

    def hrs(b):
        return (b - bjd[0]) * 24

    print(f'관측: {utc_str(bjd[0])} ~ {utc_str(bjd[-1])} UTC ({hours[-1]:.2f}시간, {len(fdf)}장)')
    print(f'식 시각 ({"예보 ±" + str(FIT_TIMING_WINDOW_MIN) + "분 안에서 탐색" if FIT_FREE_TIMING else "예보값"}, 주기 번호 {int(epoch)}):')
    for name, tt in [('식 시작', t1), ('완전히 들어감', t2), ('식 중앙', tc), ('나오기 시작', t3), ('식 끝', t4)]:
        where = '관측 중' if bjd[0] <= tt <= bjd[-1] else '관측 밖'
        print(f'  {name:7s} {utc_str(tt)} UTC  (첫 사진 기준 {hrs(tt):+.2f}시간, {where})')
    pre_min = max(0.0, (t1 - bjd[0]) * 1440)
    post_min = max(0.0, (bjd[-1] - t4) * 1440)
    print(f'식 전 기준 구간 {pre_min:.0f}분, 식 후 기준 구간 {post_min:.0f}분')
    if pre_min < 30 or post_min < 30:
        print('[주의] 식 전/후의 평평한 구간이 30분보다 짧아서, 추세와 식이 섞여 깊이가 부정확할 수 있습니다.')

    if transit_shape(bjd, tc).max() == 0:
        print('예보된 식이 관측 시간 밖에 있어 피팅하지 않습니다. 궤도 정보와 관측 날짜를 확인하세요.')
    else:
        coef, err, rms_fit, rms_flat = fit_transit(bjd, d, trend, tc)
        depth, err_m = coef[-1] * 1000, err * 1000
        snr = depth / err_m if err_m > 0 else np.nan
        print(f'\n추세({trend_name}) + 식을 동시에 맞춘 결과: 식 깊이 = {depth:+.1f} ± {err_m:.1f} mmag ({snr:.1f} 시그마), '
              f'예상 깊이 약 {EXPECTED_DEPTH_MMAG:.0f} mmag')
        print(f'{FIT_BIN_MINUTES:.0f}분 묶음 잔차: 식 포함 모델 {rms_fit * 1000:.1f} mmag / 식 없는 모델 {rms_flat * 1000:.1f} mmag')
        if snr >= 3:
            verdict = '식 신호가 통계적으로 뚜렷합니다.'
        elif snr >= 2:
            verdict = '식이 있을 가능성이 있지만 약한 신호입니다. 확정하려면 더 좋은 데이터가 필요합니다.'
        else:
            verdict = '이 데이터로는 식을 확인할 수 없습니다.'
        print('-> ' + verdict)

        loo = []
        comp_cols = sorted([c for c in fdf.columns if c.startswith('comp') and c.endswith('_mag')], key=lambda c: int(c[4:-4]))
        if len(comp_cols) >= 3 and 'target_instmag' in fdf.columns:
            M = fdf[comp_cols].values.astype(float)
            E = fdf[[c[:-4] + '_err' for c in comp_cols]].values.astype(float)
            W = np.where(np.isfinite(M) & np.isfinite(E) & (E > 0), 1.0 / np.where(E > 0, E, 1.0) ** 2, 0.0)
            Mz = np.nan_to_num(M)
            tmag = fdf['target_instmag'].values.astype(float)
            print('\n비교성을 하나씩 빼고 다시 맞춘 식 깊이 (값이 크게 바뀌면 그 비교성의 잡음 영향이 큰 것):')
            for k in range(len(comp_cols)):
                w = W.copy()
                w[:, k] = 0.0
                wsum = w.sum(1)
                cmag = np.where(wsum > 0, (Mz * w).sum(1) / np.where(wsum > 0, wsum, 1.0), np.nan)
                ck, ek, _, _ = fit_transit(bjd, tmag - cmag, trend, tc)
                loo.append(ck[-1] * 1000)
                where = COMP_XY_LIST[k] if k < len(COMP_XY_LIST) else ''
                print(f'  비교성 #{k} {where} 제외: {ck[-1] * 1000:+.1f} ± {ek * 1000:.1f} mmag')
            spread = max(loo) - min(loo)
            if spread > 2 * err_m:
                print(f'[주의] 비교성 구성에 따라 깊이가 {spread:.1f} mmag까지 달라집니다. 이 데이터의 식 깊이는 비교성 선택에 크게 '
                      '좌우되므로 믿기 어렵습니다. 원하는 값이 나오는 조합을 골라 쓰면 안 되고, 비교성은 검증 셀의 안정성 기준으로만 고르세요.')

        TRANSIT_FIT = {'depth_mmag': depth, 'err_mmag': err_m, 'snr': snr, 'tc_bjd': tc, 'trend': FIT_TREND,
                       'leave_one_out_mmag': loo}

        base_part = coef[0] + (coef[1] * trend if trend is not None else 0.0)
        resid = (d - base_part) * 1000
        bid = np.floor((bjd - bjd.min()) / (FIT_BIN_MINUTES / 1440)).astype(int)
        groups = [b for b in np.unique(bid) if (bid == b).sum() >= 3]
        bx = np.array([hours[bid == b].mean() for b in groups])
        by = np.array([np.nanmean(resid[bid == b]) for b in groups])
        be = np.array([np.nanstd(resid[bid == b]) / np.sqrt((bid == b).sum()) for b in groups])
        grid = np.linspace(bjd[0], bjd[-1], 600)

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(hours, resid, '.', ms=3, color='gray', alpha=0.4, label='each frame (trend removed)')
        ax.errorbar(bx, by, yerr=be, fmt='o', color='tab:blue', capsize=3, label=f'{FIT_BIN_MINUTES:.0f}-min bins')
        ax.plot(hrs(grid), transit_shape(grid, tc) * depth, '-', color='tab:blue', lw=2,
                label=f'fit: depth {depth:.1f} +/- {err_m:.1f} mmag')
        ax.plot(hrs(grid), transit_shape(grid, tc) * EXPECTED_DEPTH_MMAG, '--', color='crimson',
                label=f'expected: {EXPECTED_DEPTH_MMAG:.0f} mmag')
        for tt, lab in [(t1, 'T1 start'), (t2, 'T2'), (tc, 'mid'), (t3, 'T3'), (t4, 'T4 end')]:
            if bjd[0] <= tt <= bjd[-1]:
                ax.axvline(hrs(tt), color='crimson', ls=':', lw=1)
                ax.text(hrs(tt) + 0.01, 0.97, lab, color='crimson', fontsize=8, va='top', transform=ax.get_xaxis_transform())
        lo, hi = np.nanpercentile(resid, [1, 99])
        pad = max(5.0, 0.2 * (hi - lo))
        ax.set_ylim(max(hi, EXPECTED_DEPTH_MMAG) + pad, min(lo, 0.0) - pad)
        ax.set_xlabel(f'Hours since first frame (BJD_TDB {bjd[0]:.5f})')
        ax.set_ylabel('Relative brightness (mmag, up = brighter)')
        ax.set_title(f'{FIT_FILTER} filter transit fit (trend: {FIT_TREND})')
        ax.legend(fontsize=8, loc='lower left')
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()


---
## 참고 사항 / 트러블슈팅

- **로컬 런타임 연결이 안 될 때**: 터미널에서 `jupyter notebook --NotebookApp.allow_origin="https://colab.research.google.com" ...` 명령이 실행 중인지, 방화벽이 8888 포트를 막고 있지 않은지 확인하세요. 토큰 URL을 다시 복사해서 붙여넣어 보세요.
- **`'...' 에서 '...' 패턴의 파일을 찾지 못했습니다` 오류**: `BASE_DIR` 경로나 `FILE_PATTERN_BY_FILTER`의 패턴이 실제 폴더/파일명과 맞는지 확인하세요. 필터별로 폴더가 나뉜 구조라면 `BASE_DIR`을 필터마다 다르게 주는 대신, 패턴에 폴더 경로를 포함시키거나 `list_fits_files` 호출부를 필터별 폴더로 바꿔야 합니다.
- **`DATE-OBS 계열 헤더 키워드를 찾을 수 없습니다` 오류**: `header`를 직접 출력(`print(repr(header))`)해서 실제 시각 키워드 이름을 확인한 뒤 `DATE_OBS_KEYS`/`TIME_OBS_KEYS`에 추가하세요.
- **별 위치가 이미지마다 어긋나는 경우**: `ALIGN_ENABLED = True`(기본값)이면 시야의 평행이동과 수 도 이내의 회전은 자동으로 따라갑니다. "[경고] ... 별 위치 정렬에 실패" 메시지가 나오면 정렬에 쓸 밝고 고립된 별이 부족한 것이니 `DETECT_THRESHOLD_SIGMA`를 낮추거나 `ALIGN_NUM_STARS`를 늘려보세요. 180도 뒤집힘(자오선 통과 후 반전)은 자동으로 시도하지만 그 밖의 큰 회전은 지원하지 않습니다.
- **비교성 선택**: 변광성이 아닌, 시야 내 밝기가 안정적인 별을 2개 이상 고르면 앙상블 평균으로 잡음이 줄어듭니다. 3개 이상 넣으면 "(옵션) 비교성 안정성 검증" 셀이 자동으로 불안정한 별을 걸러줍니다.
- **구경 반지름**: "(옵션) 구경 반지름 자동 최적화" 셀로 실제 데이터에 맞게 최적화하세요.
- **GAIN_DEFAULT 값**: 헤더의 `EGAIN`을 믿을 수 없다면 "카메라 gain 직접 측정" 셀로 측정한 값을 씁니다.
- **아웃라이어 프레임**: "6.5 아웃라이어 프레임 자동 제거" 셀이 위치 흔들림과 오차 크기를 기준으로 자동으로 걸러냅니다.
- **추세 제거(6.6절)와 시간 비닝(6.7절)은 `TARGET_TYPE`(1=변광성/2=외계행성 식현상)으로 기본값이 자동 결정됩니다.** 변광성은 둘 다 기본 꺼짐(진짜 신호 보존), 외계행성 식현상은 둘 다 기본 켜짐(미세 신호 검출 정밀도 우선). 필요하면 설정 셀 맨 아래에서 직접 덮어쓸 수 있습니다.
- **식 곡선 피팅(11절)**: 식 깊이가 오차의 3배 이상이고, 비교성을 하나씩 빼도 비슷하게 나와야 믿을 만한 검출입니다. 식 전·후의 평평한 구간을 각각 30분~1시간 이상 찍어야 깊이가 정확해집니다. 이번 관측처럼 식 도중에 관측이 끝나면 6.6절 추세 제거(`DETRENDING_ENABLED`)는 식 신호까지 지울 수 있으니 끄고 보세요.
- **전처리(Bias/Dark/Flat)는 이 노트북 밖에서 끝내고 들어와야 합니다.**
- 이 노트북은 **기기 등급(instrumental magnitude)** 기반 차등측광입니다. 표준 등급계로 변환하려면 표준별(standard star) 관측을 이용한 영점(zero-point) 보정이 추가로 필요합니다.